In [34]:
## Librerias ##
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [35]:
listings = pd.read_csv(
    "data/Listings.csv", encoding = "ISO-8859-1", 
    low_memory=False,
    parse_dates =["host_since"]
    )

In [36]:
## informacion general del csv listings.csv ##

listings.head()
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279712 entries, 0 to 279711
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   listing_id                   279712 non-null  int64         
 1   name                         279537 non-null  object        
 2   host_id                      279712 non-null  int64         
 3   host_since                   279547 non-null  datetime64[ns]
 4   host_location                278872 non-null  object        
 5   host_response_time           150930 non-null  object        
 6   host_response_rate           150930 non-null  float64       
 7   host_acceptance_rate         166625 non-null  float64       
 8   host_is_superhost            279547 non-null  object        
 9   host_total_listings_count    279547 non-null  float64       
 10  host_has_profile_pic         279547 non-null  object        
 11  host_identity_verified    

In [37]:
## quitamos columnas innecesarias ##

columnas_eliminar = [
    "name", "host_id", "host_location", "host_response_time", "district",
    "host_response_rate", "host_acceptance_rate", "host_is_superhost", "host_has_profile_pic",
    "host_identity_verified", "amenities","maximum_nights", "review_scores_accuracy",
    "review_scores_checkin", "review_scores_communication","instant_bookable"
    ]
listings_reduce = listings.drop(columns=columnas_eliminar, errors="ignore")
print("columnas restantes:",listings_reduce.shape[1])

columnas restantes: 17


In [38]:
## vemos la cantidad de celdas vacias que contine cada columna ##
listings_reduce.isna().sum()

listing_id                       0
host_since                     165
host_total_listings_count      165
neighbourhood                    0
city                             0
latitude                         0
longitude                        0
property_type                    0
room_type                        0
accommodates                     0
bedrooms                     29435
price                            0
minimum_nights                   0
review_scores_rating         91405
review_scores_cleanliness    91665
review_scores_location       91775
review_scores_value          91785
dtype: int64

In [39]:
## Elimina las filas (no columnas) donde host_since o host_total_listings_count estén vacías. ##
listings_reduce = listings_reduce.dropna(subset=["host_since", "host_total_listings_count"])


In [40]:
## Crea una columna nueva con True si el anuncio tiene valoración global y False si está vacía. ##
listings_reduce["has_reviews"] = listings_reduce["review_scores_rating"].notna()

print(listings_reduce.shape)

(279547, 18)


In [41]:
# Nulos de bedrooms por ciudad #
(listings_reduce.groupby("city")["bedrooms"]
    .agg(nulos=lambda s: s.isna().sum(), pct=lambda s: s.isna().mean().round(3), total="size")
    .sort_values("pct", ascending=False))

,nulos,pct,total
city,,,
Paris,13394,0.2,64657
Hong Kong,1230,0.2,7083
Bangkok,2142,0.1,19360
New York,3608,0.1,36994
Istanbul,2034,0.1,24505
Cape Town,1379,0.1,19070
Sydney,2201,0.1,33596
Rio de Janeiro,1746,0.1,26591
Mexico City,809,0.0,20055


In [42]:
# ¿Los anuncios sin bedrooms son más pequeños? (si es así, probablemente son estudios)
listings_reduce.groupby(listings_reduce["bedrooms"].isna())["accommodates"].describe()

,count,mean,std,min,25%,50%,75%,max
bedrooms,,,,,,,,
False,"250,132.0",3.4,2.2,1.0,2.0,3.0,4.0,16.0
True,"29,415.0",2.3,1.2,0.0,2.0,2.0,2.0,16.0


In [43]:
# ¿Existen anuncios con 0 dormitorios? (así se marcan los estudios cuando sí se informa)
listings_reduce["bedrooms"].value_counts().sort_index().head(8)

bedrooms
1.0    170069
2.0     51355
3.0     18505
4.0      6578
5.0      2103
6.0       701
7.0       246
8.0       124
Name: count, dtype: int64

In [44]:
# Marcar qué filas se imputan
listings_reduce["bedrooms_imputed"] = listings_reduce["bedrooms"].isna()

# 1º intento: mediana por ciudad + tipo de alojamiento + capacidad
med = listings_reduce.groupby(["city", "room_type", "accommodates"])["bedrooms"].transform("median")
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].fillna(med)

# 2º intento (grupos sin datos): ciudad + capacidad
med2 = listings_reduce.groupby(["city", "accommodates"])["bedrooms"].transform("median")
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].fillna(med2)

# Último recurso: mediana global
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].fillna(listings_reduce["bedrooms"].median())

# Medianas de .5 -> entero
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].round().astype(int)

print(listings_reduce["bedrooms"].isna().sum(), "nulos restantes")
print(listings_reduce["bedrooms_imputed"].sum(), "filas imputadas")

0 nulos restantes
29415 filas imputadas


In [45]:
# 1. Eliminar precios 0 (113 filas)
listings_reduce = listings_reduce[listings_reduce["price"] > 0].copy()

# 2. Límites por ciudad: IQR sobre log10(precio), factor 3 = atípicos extremos
log_p = np.log10(listings_reduce["price"])
q = log_p.groupby(listings_reduce["city"]).quantile([.25, .75]).unstack()
iqr = q[0.75] - q[0.25]

lim = pd.DataFrame({
    "lim_inf": 10 ** (q[0.25] - 3 * iqr),
    "lim_sup": 10 ** (q[0.75] + 3 * iqr),
})

# 3. Marcar atípicos (sin borrar todavía)
aux = listings_reduce.join(lim, on="city")
listings_reduce["price_outlier"] = (aux["price"] > aux["lim_sup"]) | (aux["price"] < aux["lim_inf"])

# 4. Revisar
print(lim.round(0))
print(listings_reduce.groupby("city")["price_outlier"].agg(["sum", "mean"]).round(4))

                lim_inf  lim_sup
city                            
Bangkok            35.0 37,994.0
Cape Town          17.0 85,300.0
Hong Kong          11.0 15,837.0
Istanbul            6.0 12,042.0
Mexico City        17.0 26,498.0
New York            4.0  2,407.0
Paris               7.0  1,010.0
Rio de Janeiro      4.0 23,646.0
Rome                4.0  1,054.0
Sydney              3.0  5,677.0
                sum  mean
city                     
Bangkok          80   0.0
Cape Town        22   0.0
Hong Kong        12   0.0
Istanbul         53   0.0
Mexico City      37   0.0
New York         62   0.0
Paris           192   0.0
Rio de Janeiro   51   0.0
Rome            104   0.0
Sydney           28   0.0


In [46]:
# Quitar precios 0 y atípicos extremos (ya marcados en price_outlier)
listings_reduce = listings_reduce[~listings_reduce["price_outlier"]].copy()
listings_reduce = listings_reduce.drop(columns="price_outlier")

print(listings_reduce.shape)

(278793, 19)


In [47]:
# 1. Estadísticos por ciudad
listings_reduce.groupby("city")["minimum_nights"].describe(percentiles=[.5, .75, .95, .99])

,count,mean,std,min,50%,75%,95%,99%,max
city,,,,,,,,,
Bangkok,"19,279.0",10.5,37.0,1.0,1.0,3.0,30.0,180.0,"1,000.0"
Cape Town,"19,048.0",4.3,18.2,1.0,2.0,3.0,10.0,30.0,"1,124.0"
Hong Kong,"7,071.0",13.5,23.8,1.0,2.0,28.0,30.0,90.0,366.0
Istanbul,"24,450.0",4.5,23.0,1.0,1.0,3.0,10.0,45.0,"1,000.0"
Mexico City,"20,011.0",4.5,21.2,1.0,2.0,2.0,10.0,40.0,"1,000.0"
New York,"36,904.0",23.3,26.4,1.0,30.0,30.0,30.0,90.0,"1,250.0"
Paris,"64,403.0",5.7,44.9,1.0,2.0,4.0,20.0,89.0,"9,999.0"
Rio de Janeiro,"26,533.0",4.7,19.1,1.0,2.0,4.0,10.0,30.0,"1,000.0"
Rome,"27,526.0",3.4,16.7,1.0,2.0,3.0,6.8,30.0,"1,000.0"


In [48]:
# 2. Reparto por tramos de estancia mínima
tramos = pd.cut(
    listings_reduce["minimum_nights"],
    bins=[0, 1, 3, 7, 29, 364, float("inf")],
    labels=["1 noche", "2-3", "4-7", "8-29", "30-364 (larga estancia)", "365+"],
)
(pd.crosstab(listings_reduce["city"], tramos, normalize="index") * 100).round(1)

minimum_nights,1 noche,2-3,4-7,8-29,30-364 (larga estancia),365+
city,,,,,,
Bangkok,60.2,15.8,5.9,6.7,10.9,0.5
Cape Town,28.8,46.9,17.8,4.9,1.5,0.1
Hong Kong,44.4,12.1,5.4,27.2,10.7,0.2
Istanbul,56.3,28.6,9.1,3.3,2.6,0.2
Mexico City,47.4,38.3,8.8,2.7,2.7,0.1
New York,10.0,13.7,6.9,2.5,66.6,0.2
Paris,26.8,43.4,21.9,4.2,3.7,0.1
Rio de Janeiro,26.8,45.0,21.3,4.8,1.9,0.1
Rome,42.0,48.6,6.1,1.6,1.7,0.1


In [49]:
# 3. Valores más frecuentes (los picos suelen ser 30, 90, 365, 1000...)
listings_reduce["minimum_nights"].value_counts().head(15)

minimum_nights
1     92360
2     60776
3     37777
30    28791
5     13735
4     12838
7     11324
6      3465
10     2799
14     2327
28     1891
15     1698
29     1117
90      933
20      886
Name: count, dtype: int64

In [50]:
# 4. ¿El precio cambia con la estancia mínima? (mediana por ciudad y tramo)
(listings_reduce.groupby(["city", tramos], observed=True)["price"]
    .median().unstack().round(0))

minimum_nights,1 noche,2-3,4-7,8-29,30-364 (larga estancia),365+
city,,,,,,
Bangkok,"1,100.0","1,200.0","1,050.0","1,000.0","1,000.0","1,082.0"
Cape Town,850.0,"1,000.0","2,166.0","2,845.0",724.0,600.0
Hong Kong,421.0,538.0,590.0,264.0,320.0,600.0
Istanbul,224.0,295.0,314.0,251.0,245.0,250.0
Mexico City,613.0,737.0,650.0,600.0,500.0,"1,000.0"
New York,80.0,100.0,108.0,89.0,99.0,150.0
Paris,82.0,80.0,80.0,67.0,80.0,80.0
Rio de Janeiro,250.0,250.0,370.0,600.0,262.0,268.0
Rome,60.0,70.0,70.0,50.0,40.0,65.0


In [51]:
# Cuántos hay por encima de 365
print((listings_reduce["minimum_nights"] > 365).sum(), "anuncios con mínimo > 365 noches")

94 anuncios con mínimo > 365 noches


In [52]:
# Eliminar mínimos imposibles
listings_reduce = listings_reduce[listings_reduce["minimum_nights"] <= 365].copy()

# Marcar estancias largas (30 noches o más)
listings_reduce["long_stay"] = listings_reduce["minimum_nights"] >= 30

print(listings_reduce.shape)
print((listings_reduce.groupby("city")["long_stay"].mean() * 100).round(1).sort_values(ascending=False))

(278699, 20)
city
New York         66.8
Bangkok          11.3
Hong Kong        10.8
Paris             3.7
Mexico City       2.8
Istanbul          2.7
Sydney            2.4
Rio de Janeiro    2.0
Rome              1.7
Cape Town         1.6
Name: long_stay, dtype: float64


In [55]:
# --- ACCOMMODATES ---
# 1. Distribución general
listings_reduce["accommodates"].describe(percentiles=[.5, .9, .99])
listings_reduce["accommodates"].value_counts().sort_index()

accommodates
1      25746
2     118059
3      27872
4      57092
5      13493
6      19378
7       3681
8       6453
9       1131
10      2579
11       356
12      1028
13       192
14       322
15       198
16      1119
Name: count, dtype: int64

In [56]:
# 2. Máximo por ciudad (¿hay valores raros?)
listings_reduce.groupby("city")["accommodates"].agg(["min", "median", "max"])

,min,median,max
city,,,
Bangkok,1,2.0,16
Cape Town,1,3.0,16
Hong Kong,1,2.0,16
Istanbul,1,2.0,16
Mexico City,1,2.0,16
New York,1,2.0,16
Paris,1,2.0,16
Rio de Janeiro,1,4.0,16
Rome,1,3.0,16


In [57]:
# 3. Coherencia con dormitorios: muchos huéspedes en pocos dormitorios
incoherentes = listings_reduce[(listings_reduce["accommodates"] >= 10) & (listings_reduce["bedrooms"] <= 1)]
print(len(incoherentes), "anuncios con 10+ huéspedes y 1 dormitorio o menos")
incoherentes[["city", "room_type", "accommodates", "bedrooms", "price"]].head(10)

476 anuncios con 10+ huéspedes y 1 dormitorio o menos


,city,room_type,accommodates,bedrooms,price
28863,New York,Entire place,10,1,150
36293,Rio de Janeiro,Entire place,10,1,430
36432,Istanbul,Entire place,16,1,242
39799,New York,Entire place,10,1,140
41573,New York,Entire place,16,1,524
47266,Rio de Janeiro,Entire place,10,1,120
66374,Istanbul,Entire place,16,1,151
69128,Rio de Janeiro,Entire place,10,1,185
69863,Rio de Janeiro,Entire place,16,1,175
70805,Rio de Janeiro,Entire place,10,1,192


In [63]:
# --- PROPERTY_TYPE ---
# 4. Cuántas categorías hay y cómo se reparten
print(listings_reduce["property_type"].nunique(), "categorías")
listings_reduce["property_type"].value_counts()

31 categorías


property_type
Entire apartment                      138596
Private room in apartment              47245
Private room in house                  13279
Entire house                           13212
Entire condominium                     11219
Room in boutique hotel                  5641
Entire loft                             4577
Private room in condominium             4440
Private room in bed and breakfast       4219
Entire serviced apartment               3951
Others                                  3777
Room in hotel                           3124
Private room in townhouse               2953
Shared room in apartment                2413
Entire townhouse                        2318
Entire guest suite                      2272
Private room in serviced apartment      1539
Room in aparthotel                      1489
Entire villa                            1456
Private room in guest suite             1364
Private room in guesthouse              1330
Private room in hostel                  1

In [ ]:
### ANALISIS EXPLORATORIO DE LOS DATOS (listing_reduce.csv) ###

In [64]:
listings_exploratory = pd.read_csv(
    "data/listings_reduce.csv", encoding = "ISO-8859-1", 
    low_memory=False
    )